# 🌿 Plant Disease Detection - Training
**Model:** EfficientNetB4 + CBAM Attention Mechanism
**Baseline:** EfficientNetB4 (tanpa CBAM)

In [ ]:
!pip install tensorflow pillow matplotlib seaborn scikit-learn -q
import tensorflow as tf
print('TF Version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
import numpy as np
import os
import json
from google.colab import drive
drive.mount('/content/drive')

# Setup directories
os.makedirs('/content/drive/MyDrive/plant_disease/models', exist_ok=True)
os.makedirs('/content/drive/MyDrive/plant_disease/visualisasi', exist_ok=True)
os.makedirs('/content/drive/MyDrive/plant_disease/hasil_eksperimen', exist_ok=True)

DATASET_PATH = '/content/plantvillage/plantvillage dataset/color'
SAVE_PATH    = '/content/drive/MyDrive/plant_disease'
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 30
NUM_CLASSES  = 38
print('Setup selesai!')

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=30, width_shift_range=0.2,
    height_shift_range=0.2, horizontal_flip=True, zoom_range=0.2,
    shear_range=0.1, brightness_range=[0.8,1.2], fill_mode='nearest', validation_split=0.2
)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', seed=42
)
val_gen = train_datagen.flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', seed=42
)
print(f'Train: {train_gen.samples} | Val: {val_gen.samples}')

In [ ]:
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB4

# ── CBAM Attention Module ──
def cbam_block(x, ratio=8):
    channel = x.shape[-1]
    # Channel Attention
    avg_p = GlobalAveragePooling2D()(x)
    max_p = GlobalMaxPooling2D()(x)
    avg_p = Reshape((1,1,channel))(avg_p)
    max_p = Reshape((1,1,channel))(max_p)
    fc1 = Dense(channel//ratio, activation='relu', use_bias=False)
    fc2 = Dense(channel, use_bias=False)
    avg_out = fc2(fc1(Flatten()(avg_p)))
    max_out = fc2(fc1(Flatten()(max_p)))
    ch_att = Activation('sigmoid')(Reshape((1,1,channel))(avg_out + max_out))
    x = Multiply()([x, ch_att])
    # Spatial Attention
    avg_s = tf.reduce_mean(x, axis=-1, keepdims=True)
    max_s = tf.reduce_max(x, axis=-1, keepdims=True)
    sp_att = Conv2D(1, 7, padding='same', activation='sigmoid')(Concatenate()([avg_s, max_s]))
    x = Multiply()([x, sp_att])
    return x

# ── Baseline Model ──
def build_baseline():
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)
    return Model(base.input, out, name='Baseline_EfficientNetB4')

# ── Proposed Model ──
def build_proposed():
    inp = Input(shape=(224,224,3))
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base.trainable = False
    x = base(inp)
    x = cbam_block(x)               # CBAM Attention <-- NOVELTY
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)
    return Model(inp, out, name='Proposed_EfficientNetB4_CBAM')

baseline = build_baseline()
proposed = build_proposed()
print('Baseline params:', baseline.count_params():,)
print('Proposed params:', proposed.count_params():,)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger
import matplotlib.pyplot as plt

callbacks_baseline = [
    EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint(f'{SAVE_PATH}/models/baseline_best.h5', save_best_only=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=5, verbose=1, min_lr=1e-7),
    CSVLogger(f'{SAVE_PATH}/hasil_eksperimen/baseline_log.csv')
]
callbacks_proposed = [
    EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint(f'{SAVE_PATH}/models/proposed_best.h5', save_best_only=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=5, verbose=1, min_lr=1e-7),
    CSVLogger(f'{SAVE_PATH}/hasil_eksperimen/proposed_log.csv')
]

# ── PHASE 1: Feature Extraction ──
print('='*60)
print('TRAINING BASELINE - PHASE 1 (Feature Extraction)')
print('='*60)
baseline.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                 loss='categorical_crossentropy', metrics=['accuracy'])
history_b_p1 = baseline.fit(train_gen, validation_data=val_gen,
                             epochs=15, callbacks=callbacks_baseline)

print('='*60)
print('TRAINING PROPOSED - PHASE 1 (Feature Extraction)')
print('='*60)
proposed.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                 loss='categorical_crossentropy', metrics=['accuracy'])
history_p_p1 = proposed.fit(train_gen, validation_data=val_gen,
                             epochs=15, callbacks=callbacks_proposed)

In [ ]:
# ── PHASE 2: Fine-tuning ──
print('='*60)
print('FINE-TUNING BASELINE - PHASE 2')
print('='*60)
# Unfreeze top layers
for layer in baseline.layers[-50:]:
    if not isinstance(layer, BatchNormalization):
        layer.trainable = True
baseline.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                 loss='categorical_crossentropy', metrics=['accuracy'])
history_b_p2 = baseline.fit(train_gen, validation_data=val_gen,
                             epochs=EPOCHS, callbacks=callbacks_baseline)

print('='*60)
print('FINE-TUNING PROPOSED - PHASE 2')
print('='*60)
for layer in proposed.layers[-50:]:
    if not isinstance(layer, BatchNormalization):
        layer.trainable = True
proposed.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                 loss='categorical_crossentropy', metrics=['accuracy'])
history_p_p2 = proposed.fit(train_gen, validation_data=val_gen,
                             epochs=EPOCHS, callbacks=callbacks_proposed)

In [ ]:
# ── Plot Training Curves ──
def plot_history(history, title, path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history.history['accuracy'], 'b-', label='Train')
    ax1.plot(history.history['val_accuracy'], 'r--', label='Validation')
    ax1.set_title(f'{title} - Accuracy', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.legend()
    ax2.plot(history.history['loss'], 'b-', label='Train')
    ax2.plot(history.history['val_loss'], 'r--', label='Validation')
    ax2.set_title(f'{title} - Loss', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()

plot_history(history_b_p2, 'Baseline EfficientNetB4', f'{SAVE_PATH}/visualisasi/curve_baseline.png')
plot_history(history_p_p2, 'Proposed EfficientNetB4+CBAM', f'{SAVE_PATH}/visualisasi/curve_proposed.png')
print('Training curves disimpan!')